## Power BI Semantic Model Automation Script

In [ ]:
"""
Power BI Semantic Model Automation Script
==========================================
Automates semantic model management including object renaming, 
measure creation, calculation groups, and field parameters.

Author: Prathy Kamasani
Website: https://data-nova.com
Version: 5.3
License: MIT
"""

!pip install semantic-link-labs -qq

import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model
import re
from urllib.parse import urlparse
from typing import Optional, Dict, List
from datetime import datetime

# Configuration
POWERBI_URL = "https://app.powerbi.com/groups/c3187c38-9c45-4b7b-87c0-a9ef9b458d0b/datasets/05f26e05-560c-4c1c-82c3-dff87b29ac79/details?experience=power-bi"

def get_powerbi_ids(url: str) -> Dict[str, Optional[str]]:
    """
    Extract workspace and dataset IDs from Power BI URL.
    
    Args:
        url: Power BI dataset URL
        
    Returns:
        Dictionary containing workspace and dataset IDs
    """
    parsed = urlparse(url)
    path = parsed.path
    
    workspace_match = re.search(r'/groups/([a-f0-9-]{36})', path)
    dataset_match = re.search(r'/datasets/([a-f0-9-]{36})', path)
    
    return {
        'workspace': workspace_match.group(1) if workspace_match else None,
        'dataset': dataset_match.group(1) if dataset_match else None
    }

def run_all_operations(dataset: str, workspace: str):
    """
    Execute all semantic model operations in a single connection.
    
    Args:
        dataset: Dataset ID
        workspace: Workspace ID
        
    Returns:
        Boolean indicating success
    """
    
    print("\n" + "="*70)
    print(" "*15 + "POWER BI MODEL AUTOMATION")
    print(" "*22 + "data-nova.com")
    print("="*70)
    
    start_time = datetime.now()
    
    try:
        with connect_semantic_model(dataset=dataset, readonly=False, workspace=workspace) as tom:
            
            # Rename tables and columns
            print("\n[1/4] Renaming Objects...")
            rename_count = 0
            
            # Process table names
            for t in tom.model.Tables:
                old_name = t.Name
                new_name = t.Name.replace('_', ' ')
                for prefix in ['Dim ', 'Fact ', 'dim ', 'fact ']:
                    new_name = new_name.replace(prefix, '')
                new_name = new_name.title()
                
                if old_name != new_name:
                    t.Name = new_name
                    rename_count += 1
            
            # Process column names
            for c in tom.all_columns():
                old_name = c.Name
                new_name = c.Name.replace('_', ' ').title()
                if old_name != new_name:
                    c.Name = new_name
                    rename_count += 1
            
            print(f"   ✓ Renamed {rename_count} objects")
            
            # Create or update measures
            print("\n[2/4] Managing Measures...")
            
            # Find the "Hourly Weather" table specifically
            measure_table = None
            for table in tom.model.Tables:
                if table.Name == 'Hourly Weather':
                    measure_table = table
                    break
            
            # If not found, use first non-special table
            if not measure_table:
                for table in tom.model.Tables:
                    is_special = any(col.Name.endswith('Ordinal') for col in table.Columns)
                    if not is_special:
                        measure_table = table
                        break
            
            if measure_table:
                # Collect all existing measures across all tables
                all_existing_measures = {}
                for table in tom.model.Tables:
                    for measure in table.Measures:
                        all_existing_measures[measure.Name] = (table, measure)
                
                # Define measures to ensure exist
                measures_config = [
                    ('Avg Temperature', "AVERAGE('Hourly Weather'[Temperature 2M])", '#,0.0°C', 'Core Metrics'),
                    ('Max Temperature', "MAX('Hourly Weather'[Temperature 2M])", '#,0.0°C', 'Core Metrics'),
                    ('Min Temperature', "MIN('Hourly Weather'[Temperature 2M])", '#,0.0°C', 'Core Metrics'),
                    ('Temperature Range', "[Max Temperature] - [Min Temperature]", '#,0.0°C', 'Core Metrics'),
                    ('Record Count', f"COUNTROWS('Hourly Weather')", '#,0', 'Core Metrics')
                ]
                
                created = 0
                updated = 0
                
                for name, expression, format_string, folder in measures_config:
                    try:
                        if name in all_existing_measures:
                            # Update existing measure
                            table, measure = all_existing_measures[name]
                            measure.Expression = expression
                            measure.FormatString = format_string
                            measure.DisplayFolder = folder
                            updated += 1
                        else:
                            # Create new measure
                            tom.add_measure(
                                table_name=measure_table.Name,
                                measure_name=name,
                                expression=expression,
                                format_string=format_string,
                                display_folder=folder
                            )
                            created += 1
                    except Exception as e:
                        print(f"   ⚠️ Skipped {name}: Column may not exist")
                
                print(f"   ✓ Created {created} new, updated {updated} existing measures")
            
            # Set up calculation groups
            print("\n[3/4] Managing Calculation Groups...")
            
            # First, identify all existing calculation groups and their precedence
            existing_calc_groups = {}
            used_precedences = set()
            
            for t in tom.model.Tables:
                # Check if it's a calculation group by looking for Ordinal column
                if any(col.Name.endswith('Ordinal') for col in t.Columns):
                    existing_calc_groups[t.Name] = t
                    # Try to get precedence if accessible
                    if hasattr(t, 'CalculationGroup') and hasattr(t.CalculationGroup, 'Precedence'):
                        used_precedences.add(t.CalculationGroup.Precedence)
            
            print(f"   Found {len(existing_calc_groups)} existing calculation groups")
            
            # Remove our specific calculation groups to recreate them
            for calc_group_name in ['Time Intelligence', 'Statistical']:
                if calc_group_name in existing_calc_groups:
                    tom.remove_object(existing_calc_groups[calc_group_name])
                    print(f"   Removed existing: {calc_group_name}")
            
            # Find available precedence values
            def get_next_precedence(start_val):
                val = start_val
                while val in used_precedences:
                    val += 10
                used_precedences.add(val)
                return val
            
            # Create Time Intelligence group
            time_precedence = get_next_precedence(10)
            tom.add_calculation_group(name='Time Intelligence', precedence=time_precedence)
            
            time_calcs = [
                ("Current", "SELECTEDMEASURE()"),
                ("PY", "CALCULATE(SELECTEDMEASURE(), SAMEPERIODLASTYEAR('Date'[Date]))"),
                ("YoY", "SELECTEDMEASURE() - CALCULATE(SELECTEDMEASURE(), SAMEPERIODLASTYEAR('Date'[Date]))"),
                ("YoY %", "DIVIDE(SELECTEDMEASURE() - CALCULATE(SELECTEDMEASURE(), SAMEPERIODLASTYEAR('Date'[Date])), CALCULATE(SELECTEDMEASURE(), SAMEPERIODLASTYEAR('Date'[Date])))"),
                ("YTD", "CALCULATE(SELECTEDMEASURE(), DATESYTD('Date'[Date]))"),
                ("MTD", "CALCULATE(SELECTEDMEASURE(), DATESMTD('Date'[Date]))")
            ]
            
            for item_name, expression in time_calcs:
                try:
                    tom.add_calculation_item(
                        table_name='Time Intelligence',
                        calculation_item_name=item_name,
                        expression=expression
                    )
                except:
                    pass
            
            print(f"   ✓ Created Time Intelligence group (precedence: {time_precedence})")
            
            # Create Statistical group
            stat_precedence = get_next_precedence(30)
            tom.add_calculation_group(name='Statistical', precedence=stat_precedence)
            
            stat_items = [
                ("Value", "SELECTEDMEASURE()"),
                ("Average", "AVERAGEX(ALLSELECTED('Date'[Date]), SELECTEDMEASURE())"),
                ("Maximum", "MAXX(ALLSELECTED('Date'[Date]), SELECTEDMEASURE())"),
                ("Minimum", "MINX(ALLSELECTED('Date'[Date]), SELECTEDMEASURE())")
            ]
            
            for item_name, expression in stat_items:
                try:
                    tom.add_calculation_item(
                        table_name='Statistical',
                        calculation_item_name=item_name,
                        expression=expression
                    )
                except:
                    pass
            
            print(f"   ✓ Created Statistical group (precedence: {stat_precedence})")
            
            # Create field parameters
            print("\n[4/4] Managing Field Parameters...")
            
            # Helper function for field parameters
            def recreate_field_parameter(table_name, objects, object_names):
                """Create or recreate a field parameter table."""
                # Remove if exists
                if table_name in [t.Name for t in tom.model.Tables]:
                    tom.remove_object(tom.model.Tables[table_name])
                
                # Build DAX expression
                expr_parts = []
                for i, (obj, name) in enumerate(zip(objects, object_names)):
                    expr_parts.append(f'("{name}", NAMEOF({obj}), {i})')
                expr = "{" + ", ".join(expr_parts) + "}"
                
                # Create calculated table
                tom.add_calculated_table(name=table_name, expression=expr)
                
                # Add columns
                col2 = f"{table_name} Fields"
                col3 = f"{table_name} Order"
                tom.add_calculated_table_column(
                    table_name=table_name,
                    column_name=table_name,
                    source_column="[Value1]",
                    data_type="String",
                    hidden=False
                )
                tom.add_calculated_table_column(
                    table_name=table_name,
                    column_name=col2,
                    source_column="[Value2]",
                    data_type="String",
                    hidden=True
                )
                tom.add_calculated_table_column(
                    table_name=table_name,
                    column_name=col3,
                    source_column="[Value3]",
                    data_type="Int64",
                    hidden=True
                )
                
                # Set properties
                table = tom.model.Tables[table_name]
                table.Columns[table_name].SortByColumn = table.Columns[col3]
                table.Columns[col2].SortByColumn = table.Columns[col3]
                
                # Set field parameter metadata
                tom.set_extended_property(
                    object=table.Columns[col2],
                    extended_property_type="Json",
                    name="ParameterMetadata",
                    value='{"version":3,"kind":2}'
                )
            
            # Create Metrics parameter with existing measures
            try:
                recreate_field_parameter(
                    table_name="Metrics",
                    objects=[
                        "[Avg Temperature]",
                        "[Max Temperature]",
                        "[Min Temperature]",
                        "[Temperature Range]",
                        "[Record Count]"
                    ],
                    object_names=[
                        "Avg Temperature",
                        "Max Temperature",
                        "Min Temperature",
                        "Temperature Range",
                        "Record Count"
                    ]
                )
                print("   ✓ Created Metrics parameter")
            except Exception as e:
                print(f"   ⚠️ Could not create Metrics parameter: {str(e)[:50]}")
    
    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
        return False
    
    duration = datetime.now() - start_time
    print(f"\n{'='*70}")
    print(f"✅ Completed in {duration.total_seconds():.1f} seconds")
    print(f"{'='*70}\n")
    
    return True

# Main execution
if __name__ == "__main__":
    # Extract IDs from URL
    ids = get_powerbi_ids(POWERBI_URL)
    dataset = ids['dataset']
    workspace = ids['workspace']
    
    # Execute operations
    success = run_all_operations(dataset, workspace)
    
    # Trigger model refresh
    if success:
        print("🔄 Starting model refresh...")
        try:
            labs.refresh_semantic_model(dataset=dataset, workspace=workspace)
            print("✅ Refresh initiated")
        except Exception as e:
            print(f"⚠️ Refresh failed: {e}")

## Power BI Model Cleanup Script

In [ ]:
# """
# Power BI Model Cleanup Script - Notebook Version
# =================================================
# Removes all custom objects from the semantic model.

# Author: Prathy Kamasani
# Website: https://data-nova.com
# """

# !pip install semantic-link-labs -qq

# import sempy_labs as labs
# from sempy_labs.tom import connect_semantic_model
# import re
# from urllib.parse import urlparse

# POWERBI_URL = "https://app.powerbi.com/groups/c3187c38-9c45-4b7b-87c0-a9ef9b458d0b/datasets/05f26e05-560c-4c1c-82c3-dff87b29ac79/details?experience=power-bi"

# def get_ids(url):
#     """Extract IDs from URL"""
#     parsed = urlparse(url)
#     workspace = re.search(r'/groups/([a-f0-9-]{36})', parsed.path)
#     dataset = re.search(r'/datasets/([a-f0-9-]{36})', parsed.path)
#     return workspace.group(1), dataset.group(1)

# def cleanup_model():
#     """Remove all custom objects from the model"""
#     workspace, dataset = get_ids(POWERBI_URL)
    
#     print("\n🧹 CLEANING MODEL")
#     print("=" * 40)
    
#     with connect_semantic_model(dataset=dataset, readonly=False, workspace=workspace) as tom:
        
#         # 1. Remove all measures
#         print("\n[1/4] Removing measures...")
#         measure_count = 0
#         for table in tom.model.Tables:
#             for measure in list(table.Measures):
#                 tom.remove_object(measure)
#                 measure_count += 1
#         print(f"   ✓ Removed {measure_count} measures")
        
#         # 2. Remove calculation groups
#         print("\n[2/4] Removing calculation groups...")
#         calc_count = 0
#         for table in list(tom.model.Tables):
#             if any(col.Name.endswith('Ordinal') for col in table.Columns):
#                 tom.remove_object(table)
#                 calc_count += 1
#         print(f"   ✓ Removed {calc_count} calculation groups")
        
#         # 3. Remove field parameters
#         print("\n[3/4] Removing field parameters...")
#         param_count = 0
#         tables_to_remove = []
        
#         for table in tom.model.Tables:
#             for column in table.Columns:
#                 # Check for ParameterMetadata in any form
#                 if 'Parameter' in table.Name or 'Fields' in column.Name:
#                     if hasattr(column, 'ExtendedProperties'):
#                         tables_to_remove.append(table)
#                         break
        
#         for table in tables_to_remove:
#             try:
#                 tom.remove_object(table)
#                 param_count += 1
#             except:
#                 pass
                
#         print(f"   ✓ Removed {param_count} field parameters")
        
#         # 4. Optional: Remove specific calculated tables
#         print("\n[4/4] Removing calculated tables...")
#         calc_table_count = 0
        
#         # List specific table names to remove
#         calc_tables_to_remove = ['Metrics', 'Date Hierarchy', 'Geography Level', 
#                                  'Time Granularity', 'Weather Attributes', 
#                                  'Date Parts', 'Analysis Dimension']
        
#         for table_name in calc_tables_to_remove:
#             if table_name in [t.Name for t in tom.model.Tables]:
#                 try:
#                     tom.remove_object(tom.model.Tables[table_name])
#                     calc_table_count += 1
#                 except:
#                     pass
        
#         print(f"   ✓ Removed {calc_table_count} calculated tables")
    
#     print("\n✅ Cleanup Summary:")
#     print(f"   • {measure_count} measures")
#     print(f"   • {calc_count} calculation groups")
#     print(f"   • {param_count} field parameters")
#     print(f"   • {calc_table_count} calculated tables")
    
#     # Refresh model
#     print("\n🔄 Refreshing model...")
#     try:
#         labs.refresh_semantic_model(dataset=dataset, workspace=workspace)
#         print("   ✓ Refresh initiated")
#     except Exception as e:
#         print(f"   ⚠️ Refresh failed: {e}")

# # Run cleanup
# cleanup_model()